# Import modules

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import joblib
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Set results path

In [ ]:
results_path = '/CCLE_res'

# Functions

In [ ]:
def get_r2_mse_rmse_plot_pred(data_y_test, model_predictions, model_type):

  r2 = r2_score(data_y_test, model_predictions)
  mse = mean_squared_error(data_y_test, model_predictions)
  rmse = mse**0.5

  print(f"R-squared: {r2}")
  print(f"Mean Squared Error (MSE): {mse}")
  print(f"Root Mean Squared Error (RMSE): {rmse}")


  plt.figure(figsize=(8, 6))
  for i in range(data_y_test.shape[1]):
      plt.scatter(data_y_test.iloc[:, i], model_predictions[:, i], alpha=0.5)

  plt.plot([data_y_test.min().min(), data_y_test.max().max()], [data_y_test.min().min(), data_y_test.max().max()], 'k--', lw=2, label='Ideal')

  plt.xlabel("Actual Values")
  plt.ylabel("Predicted Values")
  plt.title(f"Predicted vs. Actual Values for Metabolites ({model_type})")
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper right')
  plt.grid(True)
  plt.savefig(f'{results_path}/{model_type}.svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
def plot_cv_scores(model_pipeline, model_type, cv_fold):

    imputer_rna = SimpleImputer(strategy='mean')  
    rna_imputed = imputer_rna.fit_transform(rna)
    rna_imputed = pd.DataFrame(rna_imputed, columns=rna.columns, index=rna.index)

    imputer_metabolites = SimpleImputer(strategy='mean')
    metabolites_imputed = imputer_metabolites.fit_transform(metabolites)
    metabolites_imputed = pd.DataFrame(metabolites_imputed, columns=metabolites.columns, index=metabolites.index)

    model_1_preds = model_1.predict(rna_imputed)
    model_2_preds = model_2.predict(rna_imputed)

    stacked_features_train = np.hstack([model_1_preds, model_2_preds])

    scores = cross_val_score(model_pipeline, stacked_features_train, metabolites_imputed, cv=cv_fold, scoring='neg_mean_squared_error', n_jobs=-1)

    print("Cross-validation scores:", scores)
    print("Average score:", scores.mean())

    scores = pd.DataFrame(scores)
    scores = scores.rename(columns={0: 'Cross-validation scores'})
    scores['Cross-validation fold'] = [i + 1 for i in range(len(scores))]

    sns.lineplot(data=scores, x='Cross-validation fold', y='Cross-validation scores')
    plt.ylabel('Negative MSE')
    plt.title('Negative MSE fluctuation with CV-fold')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.savefig(f'{results_path}/MSE vs CV ({model_type}).svg', bbox_inches='tight')
    plt.show()

In [ ]:
def get_r2_cross_val(model, model_name):
  cv = KFold(n_splits=10, shuffle=True, random_state=0)

  imputer = SimpleImputer(strategy='mean')  
  X_train_imputed = imputer.fit_transform(X_train)

  model_1_preds = model_1.predict(X_train_imputed)
  model_2_preds = model_2.predict(X_train_imputed)

  stacked_features_train = np.hstack([model_1_preds, model_2_preds])

  scores = cross_val_score(model, stacked_features_train, y_train, cv=cv, scoring='r2', n_jobs=-1)

  print("Cross-validation R-squared scores:", scores)
  print("Average R-squared score:", scores.mean())

  scores_df = pd.DataFrame({'Fold': range(1, len(scores) + 1), 'R-squared': scores})

  plt.figure(figsize=(8, 6))
  sns.lineplot(x='Fold', y='R-squared', data=scores_df, marker='o')
  plt.title(f'Cross-validation R-squared scores for {model_name}')
  plt.xlabel('Fold')
  plt.ylabel('R-squared')
  plt.savefig(f'{results_path}/R2 vs CV ({model_name}).svg', bbox_inches='tight')
  plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error

def get_metrics(y_predicted):

  rmse = mean_squared_error(y_test, y_predicted)
  print("RMSE:", rmse)

  from sklearn.metrics import mean_absolute_error
  mae = mean_absolute_error(y_test, y_predicted)
  print("MAE:", mae)

  from sklearn.metrics import median_absolute_error
  medae = median_absolute_error(y_test, y_predicted,)
  print("MedAE:", medae)

  def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

  mape = mean_absolute_percentage_error(y_test, y_predicted)
  print('Mean absolute percentage error:', mape)

In [ ]:
from scipy.stats import spearmanr

def get_spearman_correlation(model_predictions, model_name):

  correlation, p_value = spearmanr(X_test, model_predictions)

  correlation_df = pd.DataFrame({'Actual': y_test.values.flatten(), 'Predicted': model_predictions.flatten()})

  
  correlation, p_value = spearmanr(correlation_df['Actual'], correlation_df['Predicted'])

 
  correlation_df['Correlation'] = correlation
  correlation_df['P-value'] = p_value

  sns.regplot(x='Actual', y='Predicted', data=correlation_df, line_kws={'color': 'red'})
  plt.title(f'Spearman\'s Correlation ({model_name})')
  plt.xlabel('Actual Values')
  plt.ylabel('Predicted Values')


  plt.text(0.1, 0.9, f'Correlation: {correlation:.2f}\nP-value: {p_value:.2f}', transform=plt.gca().transAxes)
  plt.savefig(f'{results_path}/Spearmans Corr ({model_name}).svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
#Checking for heteroscedasticity
!pip install statsmodels

import statsmodels.api as sm

def get_residuals_vs_pred_plot(model_predictions, model_name):

  residuals = (y_test - model_predictions).values.ravel()

  plt.figure(figsize=(8, 6))
  plt.scatter(model_predictions.ravel(), residuals, alpha=0.6)
  plt.axhline(y=0, color='red', linestyle='--', linewidth=1)
  plt.title(f"Residuals vs. Predicted Values ({model_name})")
  plt.xlabel("Predicted Values")
  plt.ylabel("Residuals")
  plt.savefig(f'{results_path}/Residuals vs Pred Values ({model_name}).svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
def plot_r2_per_metabolite(model, model_name, X_test, y_test, model_1, model_2):
    """Plots the R2 score per metabolite for a given model in descending order."""

    
    imputer = SimpleImputer(strategy='mean')
    X_test_imputed = imputer.fit_transform(X_test)

    
    model_1_preds_test = model_1.predict(X_test_imputed)
    model_2_preds_test = model_2.predict(X_test_imputed)
    stacked_features_test = np.hstack([model_1_preds_test, model_2_preds_test])

    y_pred = model.predict(stacked_features_test)

    r2_scores = []
    for i in range(y_test.shape[1]):
        r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
        r2_scores.append(r2)

    
    r2_df = pd.DataFrame({'Metabolite': y_test.columns, 'R2 Score': r2_scores})

    
    r2_df = r2_df.sort_values(by=['R2 Score'], ascending=False)

    
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(r2_df)), r2_df['R2 Score'])
    plt.xticks()
    plt.xlabel("Metabolite")
    plt.ylabel("R2 Score")
    plt.title(f"R2 Score per Metabolite ({model_name}) - Descending Order")
    plt.tight_layout()
    plt.savefig(f'{results_path}/R2 per metabolite ({model_name}).svg', bbox_inches = 'tight')
    plt.show()

In [ ]:
def get_r2_per_metabolite(model, model_name, X_test, y_test):
    """Get the R2 score per metabolite for a given model in descending order."""

    imputer = SimpleImputer(strategy='mean')
    X_test_imputed = imputer.fit_transform(X_test)
    
    model_1_preds_test = model_1.predict(X_test_imputed)
    model_2_preds_test = model_2.predict(X_test_imputed)
    stacked_features_test = np.hstack([model_1_preds_test, model_2_preds_test])

    y_pred = model.predict(stacked_features_test)

    r2_scores = []
    for i in range(y_test.shape[1]):
        r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
        r2_scores.append(r2)

    
    r2_df = pd.DataFrame({'Metabolite': y_test.columns, 'R2 Score': r2_scores})

   
    r2_df = r2_df.sort_values(by=['R2 Score'], ascending=False)

    return r2_df

In [ ]:
from scipy.stats import spearmanr

def plot_spearman_per_metabolite(model, model_name, X_test, y_test):
    """Plots the Spearman's correlation coefficient per metabolite for a given model."""
    
    imputer = SimpleImputer(strategy='mean')
    X_test_imputed = imputer.fit_transform(X_test)

    model_1_preds_test = model_1.predict(X_test_imputed)
    model_2_preds_test = model_2.predict(X_test_imputed)
    stacked_features_test = np.hstack([model_1_preds_test, model_2_preds_test])

    y_pred = model.predict(stacked_features_test) 

    spearman_coeffs = [] 
    for i in range(y_test.shape[1]):  
        coeff, _ = spearmanr(y_test.iloc[:, i], y_pred[:, i]) 
        spearman_coeffs.append(coeff)  

    spearman_df = pd.DataFrame({'Metabolite': y_test.columns, 'Spearman Coefficient': spearman_coeffs})

    spearman_df = spearman_df.sort_values(by=['Spearman Coefficient'], ascending=False)

    # Plotting
    plt.figure(figsize=(10, 6))  
    plt.bar(range(len(spearman_df)), spearman_df['Spearman Coefficient'])  
    plt.xticks() 
    plt.xlabel("Metabolite")
    plt.ylabel("Spearman's Coefficient")
    plt.title(f"Spearman's Correlation Coefficient per Metabolite ({model_name}) - Descending Order")
    plt.tight_layout() 
    plt.savefig(f'{results_path}/Spearman per metabolite ({model_name}).svg', bbox_inches = 'tight')
    plt.show()

In [ ]:
def get_spearman_per_metabolite(model, model_name, X_test, y_test):
    """Get the Spearman's correlation coefficient per metabolite for a given model."""
    
    imputer = SimpleImputer(strategy='mean')
    X_test_imputed = imputer.fit_transform(X_test)

    model_1_preds_test = model_1.predict(X_test_imputed)
    model_2_preds_test = model_2.predict(X_test_imputed)
    stacked_features_test = np.hstack([model_1_preds_test, model_2_preds_test])

    y_pred = model.predict(stacked_features_test)  

    spearman_coeffs = [] 
    for i in range(y_test.shape[1]):  
        coeff, _ = spearmanr(y_test.iloc[:, i], y_pred[:, i]) 
        spearman_coeffs.append(coeff) 

    spearman_df = pd.DataFrame({'Metabolite': y_test.columns, 'Spearman Coefficient': spearman_coeffs})

    spearman_df = spearman_df.sort_values(by=['Spearman Coefficient'], ascending=False)

    return spearman_df

In [ ]:
def save_model(model, model_name):
    joblib.dump(model, f'{results_path}/{model_name}.pkl')

# Import data

In [ ]:
rna = pd.read_csv('CCLE_transcriptomics_tpm_metabolism_related_genes.csv', index_col = 0 )
rna

In [ ]:
metabolites = pd.read_csv('CCLE_metabolites.csv', index_col = 0)
metabolites

In [ ]:
threshold = 0.2
nan_percentage = metabolites.isnull().mean()
filtered_metabolites = metabolites.loc[:, nan_percentage <= threshold]
metabolites = filtered_metabolites
metabolites

# Remove outliers

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer

def remove_outliers_isolationforest(data, contamination=0.05):
    """Removes outliers using Isolation Forest."""

    imputer = SimpleImputer(strategy='mean') 
    data_clean = imputer.fit_transform(data)

    iso = IsolationForest(contamination=contamination)
    yhat = iso.fit_predict(data_clean)
    filtered_data = data[yhat != -1] 
    return filtered_data

filtered_rna = remove_outliers_isolationforest(rna)
filtered_metabolites = remove_outliers_isolationforest(metabolites)

In [ ]:
rna = filtered_rna
metabolites = filtered_metabolites

In [ ]:
rna

# Data shuffling

In [ ]:
combined_data = pd.concat([rna, metabolites], axis=1)
shuffled_data = combined_data.sample(frac=1, random_state=0)
shuffled_rna = shuffled_data[rna.columns]
shuffled_metabolites = shuffled_data[metabolites.columns]

rna = shuffled_rna
metabolites = shuffled_metabolites

# Log2 transformation

In [ ]:
#CCLE data are already log2 transformed
# rna = rna.apply(lambda x: np.log2(x + 1))
# rna.head(3)

# Splitting data

In [ ]:
metabolites = metabolites.dropna(axis=1, how='all')
metabolites

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(rna, metabolites, test_size=0.2, random_state=0)

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer_train = SimpleImputer(strategy='mean')
y_train_imputed = imputer_train.fit_transform(y_train)

In [ ]:
imputer_test = SimpleImputer(strategy='mean')
y_test_imputed = imputer_test.fit_transform(y_test)

In [ ]:
y_train = pd.DataFrame(y_train_imputed, columns=y_train.columns, index=y_train.index)
y_test = pd.DataFrame(y_test_imputed, columns=y_test.columns, index=y_test.index)

In [ ]:
if y_train.isnull().any().any():
    print("DataFrame contains NaN values")
else:
    print("DataFrame does not contain NaN values")

In [ ]:
if y_test.isnull().any().any():
    print("DataFrame contains NaN values")
else:
    print("DataFrame does not contain NaN values")

In [ ]:
y_train_features = y_train.columns
y_train_features

# Ensemble training

In [ ]:
'''
Ensemble architecture:

base models: elastic net + random forest regressor
final estimator: ridge regression

'''

In [ ]:
model_1 = joblib.load('ElasticNet.pkl')

In [ ]:
model_2 = joblib.load('Random Forest Regressor.pkl')

In [ ]:
model_1_preds = model_1.predict(X_train)
model_2_preds = model_2.predict(X_train)

stacked_features_train = np.hstack([model_1_preds, model_2_preds])

In [ ]:
param_grid = {'alpha': [0.1, 1, 2, 4, 6, 8, 10, 50, 100, 110, 125, 130, 140, 150, 175, 200]}

ridge = Ridge(random_state = 0)
grid_search = GridSearchCV(ridge, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(stacked_features_train, y_train)

final_estimator = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

model_1_preds_test = model_1.predict(X_test)
model_2_preds_test = model_2.predict(X_test)

stacked_features_test = np.hstack([model_1_preds_test, model_2_preds_test])
y_pred = final_estimator.predict(stacked_features_test)

In [ ]:
save_model(model = final_estimator, model_name = 'Ensemble')

# Model assessment

In [ ]:
get_r2_mse_rmse_plot_pred(data_y_test = y_test, model_predictions = y_pred, model_type = 'Ensemble')

In [ ]:
plot_cv_scores(model_pipeline = final_estimator, model_type = 'Ensemble', cv_fold = 10)

In [ ]:
get_r2_cross_val(model = final_estimator, model_name = 'Ensemble')

In [ ]:
get_metrics(y_predicted = y_pred)

In [ ]:
get_spearman_correlation(model_predictions = y_pred, model_name = 'Ensemble')

In [ ]:
get_residuals_vs_pred_plot(model_predictions = y_pred, model_name = 'Ensemble')

In [ ]:
plot_r2_per_metabolite(final_estimator, "Ensemble", X_test, y_test, model_1, model_2)

In [ ]:
r2_ensemble = get_r2_per_metabolite(final_estimator, 'Ensemble', X_test, y_test)
r2_ensemble

In [ ]:
plot_spearman_per_metabolite(final_estimator, 'Ensemble', X_test, y_test)

In [ ]:
spearman_ensemble = get_spearman_per_metabolite(final_estimator, 'Ensemble', X_test, y_test)
spearman_ensemble

In [ ]:
r2_ensemble.to_csv(f'{results_path}/r2_ensemble.csv', index = False)
spearman_ensemble.to_csv(f'{results_path}/spearman_ensemble.csv', index = False)